# Modularity Null Model (Weighted)

Benchmarks the modularity of the canonical Louvain partition (Q = 0.7991) against randomized null networks (R2 comment 3.2, Table S13).

* Graph: undirected, reciprocal edge weights summed (as used for community detection).
* Degree-preserving null: `double_edge_swap` with ten swaps per edge, edge weights permuted across rewired edges, 100 randomizations (seeds 0 to 99).
* Erdos-Renyi null: G(n, m) with the same number of nodes and edges, as a reference for the modularity produced by sparsity alone.
* Each null network is partitioned with the same Louvain settings (python-louvain, resolution 1.0, random_state 42).
* Run on the full network and on its largest connected component.

Produces the weighted and Erdos-Renyi rows of Table S13. The unweighted rows come from `Modularity Null Model (Unweighted).py`.

Inputs (from `derived-data/hashed/`): `network_edges_hashed.csv`, `network_nodes_with_community_hashed.csv`.

In [1]:
!pip -q install python-louvain
import pandas as pd, numpy as np, networkx as nx, time
import community as community_louvain
print('networkx', nx.__version__)

def norm(s): return s.astype(str).str.strip().str.lstrip('@').str.lower()

edges = pd.read_csv('network_edges_hashed.csv')            # derived-data/hashed/
edges['source'], edges['target'] = norm(edges['source']), norm(edges['target'])
nodes = pd.read_csv('network_nodes_with_community_hashed.csv', sep=';', encoding='utf-8-sig')  # derived-data/hashed/
nodes['username'] = norm(nodes['username'])

# Undirected weighted graph used for Louvain: reciprocal edge weights summed
Gu = nx.Graph()
for s, t, w in edges[['source','target','weight']].itertuples(index=False):
    if s == t: continue
    if Gu.has_edge(s, t): Gu[s][t]['weight'] += w
    else: Gu.add_edge(s, t, weight=w)
print('nodes', Gu.number_of_nodes(), '| undirected edges', Gu.number_of_edges())

networkx 3.7
nodes 11426 | undirected edges 13923


In [2]:
part_canon = dict(zip(nodes['username'], nodes['community']))
print('nodes without community:', len(set(Gu) - set(part_canon)))
Q_canon = community_louvain.modularity(part_canon, Gu, weight='weight')
part42 = community_louvain.best_partition(Gu, weight='weight', resolution=1.0, random_state=42)
Q42 = community_louvain.modularity(part42, Gu, weight='weight')
print(f'Q_canon={Q_canon:.4f} | Q_seed42={Q42:.4f} | communities={len(set(part42.values()))}')

comps = [len(c) for c in nx.connected_components(Gu)]
print('connected components:', len(comps), '| LCC:', max(comps),
      f'({max(comps)/Gu.number_of_nodes():.1%})', '| two-node components:', sum(c == 2 for c in comps))

nodes without community: 0
Q_canon=0.7991 | Q_seed42=0.7926 | communities=1669
connected components: 1616 | LCC: 7137 (62.5%) | two-node components: 1166


In [3]:
def null_Q(G, kind, s):
    w = np.array([d['weight'] for *_, d in G.edges(data=True)])
    rng = np.random.default_rng(s)
    if kind == 'config':   # degree-preserving rewiring
        H = nx.Graph(); H.add_nodes_from(G); H.add_edges_from(G.edges())
        E = H.number_of_edges()
        nx.double_edge_swap(H, nswap=10*E, max_tries=1000*E, seed=s)
    else:                  # Erdos-Renyi G(n, m)
        H = nx.gnm_random_graph(G.number_of_nodes(), G.number_of_edges(), seed=s)
    for (u, v), wi in zip(H.edges(), rng.permutation(w)):
        H[u][v]['weight'] = float(wi)
    p = community_louvain.best_partition(H, weight='weight', resolution=1.0, random_state=42)
    return community_louvain.modularity(p, H, weight='weight')

def run(G, Q_obs, kind, n=100, label=''):
    t = time.time()
    Q = np.array([null_Q(G, kind, s) for s in range(n)])
    z = (Q_obs - Q.mean()) / Q.std(ddof=1)
    p = (np.sum(Q >= Q_obs) + 1) / (n + 1)
    print(f'{label} {kind}: Q_obs={Q_obs:.4f} | Q_null={Q.mean():.4f} ± {Q.std(ddof=1):.4f} '
          f'| dQ={Q_obs - Q.mean():.4f} | z={z:.1f} | p={p:.4f} | {time.time()-t:.0f}s')
    return Q

In [4]:
# Full network: degree-preserving and Erdos-Renyi nulls (100 randomizations each)
Qc  = run(Gu, Q_canon, 'config', 100, 'FULL')
Qe  = run(Gu, Q_canon, 'er',     100, 'FULL')

FULL config: Q_obs=0.7991 | Q_null=0.7755 ± 0.0019 | dQ=0.0236 | z=12.8 | p=0.0099 | 263s
FULL er: Q_obs=0.7991 | Q_null=0.8259 ± 0.0014 | dQ=-0.0268 | z=-19.4 | p=1.0000 | 202s


In [5]:
# Largest connected component: same nulls
L  = Gu.subgraph(max(nx.connected_components(Gu), key=len)).copy()
pL = community_louvain.best_partition(L, weight='weight', resolution=1.0, random_state=42)
QL = community_louvain.modularity(pL, L, weight='weight')
QLc = run(L, QL, 'config', 100, 'LCC')
QLe = run(L, QL, 'er',     100, 'LCC')

pd.DataFrame({'full_config': Qc, 'full_er': Qe, 'lcc_config': QLc, 'lcc_er': QLe}) \
  .to_csv('modularity_null_model.csv', index=False)

LCC config: Q_obs=0.7257 | Q_null=0.6797 ± 0.0020 | dQ=0.0460 | z=22.7 | p=0.0099 | 189s
LCC er: Q_obs=0.7257 | Q_null=0.7483 ± 0.0013 | dQ=-0.0226 | z=-17.7 | p=1.0000 | 122s


In [6]:
# Check: number of communities inside the largest component, and whether any other component is split
comps = list(nx.connected_components(Gu))
L_nodes = max(comps, key=len)
print('communities in largest component:', len({part_canon[n] for n in L_nodes}))
print('non-largest components split into >1 community:',
      sum(len({part_canon[n] for n in c}) > 1 for c in comps if len(c) != len(L_nodes)))

communities in largest component: 52
non-largest components split into >1 community: 0
